In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence, RunnableLambda, RunnableParallel


# 1. Explicitly load the variables from your .env file
load_dotenv()

# 2. (Optional) Run a quick sanity print statement to check it loaded
print("Loaded Key prefix:", os.environ.get("OPENAI_API_KEY")[:6])

# 3. Initialize your LangChain architecture
llm_openai = init_chat_model(model="gpt-4o-mini", model_provider="openai", temperature=0)

# ***SQL DATABASE AGENT***

In [ ]:
from langchain_community.utilities.sql_database import SQLDatabase

In [ ]:
sql_db = SQLDatabase.from_uri("sqlite:///SALES_DB/sales.db")

In [ ]:
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(db=sql_db, llm=llm_openai)


In [ ]:
toolkit.get_tools()

In [ ]:
from langchain.agents import create_agent

agent = create_agent(llm_openai, toolkit.get_tools())
agent

In [ ]:
example_query = str(input("Enter your query:"))

events = agent.stream(
    {"messages": [("user", example_query)]},
    {"messages": [("system", "You are a friendly assistant")]},
    stream_mode="values",
)

for event in events:
    event["messages"][-1].pretty_print()